# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import base64
import numpy as np
import wave
import io


In [ ]:
# Constants I wont image model due due to costs.
CHAT_MODEL = "openai/gpt-5-mini"
#IMAGE_MODEL = "openai/gpt-5-image-mini" 
AUDIO_MODEL = "openai/gpt-audio-mini"

In [ ]:
# set up environment
load_dotenv(override=True)

openai_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_url = os.getenv('OPENROUTER_API_URL')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    print(f"OpenAI API Url exists and is {openai_api_url}")
else:
    print("OpenAI API Key not set")
    
client = OpenAI(
    base_url=openai_api_url, 
    api_key=openai_api_key,
)




In [ ]:
# Chat messages

system_message = """
You are a helpful technical expert for Q&A forum.
Give short, courteous answers, no more than 10 sentence.
Always be accurate. If you don't know the answer, say so.
"""


In [ ]:
def talker(message):
    stream = client.chat.completions.create(
        model=AUDIO_MODEL,
        modalities=["text", "audio"],
        audio={"voice": "alloy", "format": "pcm16"},
        stream=True,
        messages=[{"role": "user", "content": message}]
    )

    full_audio_base64 = ""
    for chunk in stream:
        if hasattr(chunk.choices[0].delta, 'audio') and chunk.choices[0].delta.audio:
            if 'data' in chunk.choices[0].delta.audio:
                full_audio_base64 += chunk.choices[0].delta.audio['data']

    if not full_audio_base64:
        return None

    audio_bytes = base64.b64decode(full_audio_base64)
    if len(audio_bytes) % 2 != 0:
        audio_bytes += b'\x00'

    audio_data = np.frombuffer(audio_bytes, dtype=np.int16)
    return (24000, audio_data)

In [ ]:
def process_input(text_input, audio_input, history):
    user_content = []
    
    if text_input and isinstance(text_input, str) and text_input.strip():
        user_content.append({"type": "text", "text": text_input})
    
    if audio_input is not None:
        sr, y = audio_input
        buffer = io.BytesIO()
        with wave.open(buffer, "wb") as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(sr)
            wav_file.writeframes(y.tobytes())
        base64_audio = base64.b64encode(buffer.getvalue()).decode("utf-8")
        user_content.append({
            "type": "input_audio",
            "input_audio": {"data": base64_audio, "format": "wav"}
        })
    
    if not user_content:
        print("No input provided.")
        return history, None
    
    messages = [{"role": "system", "content": system_message}]
    for h in history:
        messages.append({"role": h["role"], "content": h["content"]})
    
    user_message_text = text_input if (text_input and isinstance(text_input, str) and text_input.strip()) else "[Audio message]"
    
    if audio_input is not None:
        messages.append({"role": "user", "content": user_content})
        
        stream = client.chat.completions.create(
            model=AUDIO_MODEL,
            modalities=["text", "audio"],
            audio={"voice": "alloy", "format": "pcm16"},
            stream=True,
            messages=messages
        )
        
        full_text = ""
        full_audio_base64 = ""
        
        for chunk in stream:
            if hasattr(chunk.choices[0].delta, 'content') and chunk.choices[0].delta.content:
                full_text += chunk.choices[0].delta.content
            if hasattr(chunk.choices[0].delta, 'audio') and chunk.choices[0].delta.audio:
                if 'data' in chunk.choices[0].delta.audio:
                    full_audio_base64 += chunk.choices[0].delta.audio['data']
        
        if not full_text:
            full_text = "I received your audio message."
        
        if full_audio_base64:
            audio_bytes = base64.b64decode(full_audio_base64)
            if len(audio_bytes) % 2 != 0:
                audio_bytes += b'\x00'
            audio_data = np.frombuffer(audio_bytes, dtype=np.int16)
            audio_output = (24000, audio_data)
        else:
            audio_output = None
        
        updated_history = history + [
            {"role": "user", "content": user_message_text},
            {"role": "assistant", "content": full_text}
        ]
        
        return updated_history, audio_output
    
    else:
        messages.append({"role": "user", "content": text_input})
        
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages
        )
        
        full_text = response.choices[0].message.content
        audio_output = talker(full_text)
        
        updated_history = history + [
            {"role": "user", "content": text_input},
            {"role": "assistant", "content": full_text}
        ]
        
        return updated_history, audio_output

In [ ]:
# UI definition

def clear_audio():
    return None


with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")

    with gr.Row():
        audio_output = gr.Audio(streaming=True, autoplay=True, label="AI Voice Response")

    with gr.Row():
        with gr.Column():
            message = gr.Textbox(label="Type your message", placeholder="Or use the mic below...")
            submit_text = gr.Button("Send text")
            mic_in = gr.Audio(sources=["microphone"], type="numpy", label="Record Audio")
            submit_audio = gr.Button("Send audio")

    submit_audio.click(
        process_input, inputs=[message, mic_in, chatbot], outputs=[chatbot, audio_output]
    ).then(lambda: "", outputs=message)
    
    submit_text.click(
        clear_audio, outputs=mic_in
    ).then(
        process_input, inputs=[message, mic_in, chatbot], outputs=[chatbot, audio_output]
    ).then(lambda: "", outputs=message)

ui.launch(inbrowser=True, auth=("user", "pass"))
